# Developing an Image Classification AI Model

In this project, you'll train an image classifier on a dataset of your choice. Your task will be to fill the missing parts in this notebook and train a deep convolutional neural network on this dataset.

First we will need to import all the necessary libraries.

In [ ]:
# Package imports.

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sb

import os
import json
from time import time
from PIL import Image

import torch
from torch import nn, optim
import torch.nn.functional as F
from torchvision import datasets, transforms, models

Next we set some variables like the directories where our images can be found. Your tasks:

- Save the directory where your train/valid/test directories can be found
- Specify a number of epochs
- Specify the batch size

In [ ]:
# Parameters for training process
BATCH_SIZE=16
IMG_SHAPE=224
EPOCHS=25
MODEL_NAME="mobilenetv3"
FINE_TUNING=True
BASE_DIR = os.getenv("DL_BASE_DIR") if os.getenv("DL_BASE_DIR") else "images/flowers"
TRAIN_DIR = os.getenv("DL_DIR_TRAIN") if os.getenv("DL_DIR_TRAIN") else "train"
VAL_DIR = os.getenv("DL_DIR_VAL") if os.getenv("DL_DIR_VAL") else "valid"
TEST_DIR = os.getenv("DL_DIR_TEST") if os.getenv("DL_DIR_TRAIN") else "test"

# Checking if the 'GPU' is available to pass it for the device variable, and if it's not, pass the 'CPU'
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print("- Training dir is {}\n- Validation dir is {}\n- Test dir is {}".format(TRAIN_DIR, VAL_DIR, TEST_DIR))
print("CUDA is available" if torch.cuda.is_available() else "Training on CPU")

Now let's print some information about the train / valid / test directories

In [ ]:
train_dir = os.path.join(BASE_DIR, TRAIN_DIR)
valid_dir = os.path.join(BASE_DIR, VAL_DIR)
test_dir = os.path.join(BASE_DIR, TEST_DIR)

categories = list(map(lambda x : {"name": x, "dir": x}, os.listdir(train_dir)))
n_categories = len(categories)

train_dirs = []
valid_dirs = []
test_dirs = []

for i, category in enumerate(categories):
    train_dirs.append(os.path.join(train_dir, category["dir"]))
    valid_dirs.append(os.path.join(valid_dir, category["dir"]))
    test_dirs.append(os.path.join(test_dir, category["dir"]))

In [ ]:
print('The dataset contains:')
for i, category in enumerate(categories):
    print('\u2022 {}: {:,} training images, {:,} validation images and {:,} test images'.format(category["name"], len(os.listdir(train_dirs[i])), len(os.listdir(valid_dirs[i])), len(os.listdir(test_dirs[i]))))

## Load the data 

In [ ]:
# Data augmentation and Normalization.

# Defining transforms for the training, validation, and testing sets
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize(256), # resize to 256x256 pixels
        transforms.RandomRotation(30), # random rotation
        transforms.RandomResizedCrop(244), # random resized crop of 224
        transforms.RandomHorizontalFlip(), # random horizontal flip
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ]),
    'valid': transforms.Compose([
        transforms.Resize(256), # resize to 256x256 pixels
        transforms.RandomResizedCrop(244), # random resized crop of 224
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225])
    ])
}

# Loading the datasets with ImageFolder.
image_datasets = {
    'train': datasets.ImageFolder(train_dir, transform=data_transforms['train']), # create an ImageFolder dataset for the train directory
    'valid': datasets.ImageFolder(valid_dir, transform=data_transforms['valid']), # create an ImageFolder dataset for the valid directory
    'test': datasets.ImageFolder(test_dir, transform=data_transforms['valid'])  # create an ImageFolder dataset for the test directory
}

# Using the image datasets and the transforms to define the dataloaders.
dataloaders = {
    'trainloader': torch.utils.data.DataLoader(image_datasets['train'], batch_size=BATCH_SIZE, shuffle=True), # create a dataloader for the train dataset
    'validloader': torch.utils.data.DataLoader(image_datasets['valid'], batch_size=BATCH_SIZE), # create a dataloader for the train dataset
    'testloader': torch.utils.data.DataLoader(image_datasets['test'], batch_size=BATCH_SIZE) # create a dataloader for the train dataset
}

### Label mapping

You will also need to create a JSON file mapping the class IDs to the labels of your dataset. The content of this label file could look like this:

```json
{
    0: "cat",
    1: "dog",
}
```

Each label gets assigned an integer value, starting with 0 for the first category. Use alphabetic order for the labels in the JSON file.

In [ ]:
with open(BASE_DIR + '/labels.json', 'r') as f:
    cat_to_name = json.load(f)

# Building and training the classifier

Now that we have defined the dataset and the dataloader we can start building the classification network. We will start building the classification head, which will map the output of the convolutional network to the classes.

In [ ]:
# Defining a class for the classification head network.
class Classifier(nn.Module):
    def __init__(self, input_size, output_size, hidden_layers, drop_p=0.5):
        ''' Builds a feedforward network with arbitrary hidden layers.
        
            Arguments
            ---------
            input_size: integer, size of the input layer
            output_size: integer, size of the output layer
            hidden_layers: list of integers, the sizes of the hidden layers
        '''
        super().__init__()
        # Add the first layer, input to a hidden layer.
        self.hidden_layers = nn.ModuleList([nn.Linear(input_size, hidden_layers[0])])
        
        # Add a variable number of more hidden layers.
        layer_sizes = zip(hidden_layers[:-1], hidden_layers[1:])
        self.hidden_layers.extend([nn.Linear(h1, h2) for h1, h2 in layer_sizes])
        
        self.output = nn.Linear(hidden_layers[-1], output_size)
        
        self.dropout = nn.Dropout(p=drop_p)
        
    def forward(self, x):
        ''' Forward pass through the network, returns the output log softmax.
            Arguments
            ---------
            self: all layers
            x: tensor vector
        '''
        
        # Forward through each layer in `hidden_layers`, with ReLU activation and dropout.   
        # fill the missing gaps     
        for h in self.hidden_layers:
            x = F.relu(h(x))
            x = self.dropout(x)
        x = self.output(x)
        
        return F.log_softmax(x, dim=1)

### Validation Function

In the next cell we define a validation function that computes metrics like the loss and the accuracy of a model. You do not need to implement anything here, but you should have a look at the function either.

In [ ]:
# Definig a function for the validation pass.
def validation(model, validloader, criterion, device):
    ''' Builds a feedforward network with arbitrary hidden layers, 
        returns the validation loss and  validation accuracy.
        
        Arguments
        ---------
        model: the pre-trained model.
        validloader: generator, the validation dataset.
        criterion: loss function.
        device: the used device for the training [GPU or CPU].
    '''
    # Initiate the validation accuracy & validation loss with 0.
    valid_accuracy = 0
    valid_loss = 0
    # Move model to the device
    model.to(device)
    
    # Looping through the data batches.
    for inputs, labels in validloader:
        # Move input and label tensors to the device.
        inputs, labels = inputs.to(device), labels.to(device)
        
        # Forward pass through the network.
        output = model.forward(inputs)
        # Increase the validation loss by the loss of the predicted output with the labels.
        valid_loss += criterion(output, labels).item()

        ## Calculating the accuracy 
        # Model's output is log-softmax, so take exponential to get the probabilities.
        ps = torch.exp(output)
        # Class with highest probability is our predicted class, compare with true label.
        equality = (labels.data == ps.max(dim=1)[1])
        # Accuracy is number of correct predictions divided by all predictions, so we just take the mean.
        valid_accuracy += equality.type(torch.FloatTensor).mean()

    return valid_loss, valid_accuracy

### Training Function

We also create a function for training a model. This function accepts the model itself, a criterion (which is the loss function) and an optimizer. Also it takes the dataloaders for the training and validation datasets. The parameter `epochs` defines how many epochs the model is trained. With `print_every` you can specify how often the progress of the learning process is printed to the console.

In [ ]:
# Definig a function for the training process
def training(model, criterion, optimizer, device, trainloader, validloader, epochs=5, print_every=10):
    ''' Builds a feedforward network with arbitrary hidden layers.
        
        Arguments
        ---------
        model: the pre-trained model.
        optimizer: which we will take a step with it to update the weights.
        criterion: loss function.
        device: the used device for the training [GPU or CPU].
        trainloader: generator, the training dataset.
        validloader: generator, the validation dataset.
        epochs: integer, number of trainings.
        print_every: integer, printing the updates on loss & accuracy every print_every value.
    '''
    
    # Initialize steps and running_loss variables
    steps = 0
    running_loss = 0

    # Move model to the device
    model.to(device)
    
    print(f"Start training model on {device} ...")
    
    # Training for some epochs starts here
    for e in range(epochs):
        # Model in training mode, dropout is on.
        model.train()

        # Iterate over the dataloader
        for inputs, labels in trainloader:
            # Move input and label tensors to the device.
            inputs, labels = inputs.to(device), labels.to(device)
            
            steps += 1
            # zero-ing the accumalated gradients.
            optimizer.zero_grad()
            
            # Forward pass through the network
            output = model.forward(inputs)
            # Calculate the loss
            loss = criterion(output, labels)
            # Backward pass through the network 
            loss.backward()
            # Take a step with the optimizer to update the weights
            optimizer.step()
            # Update the running loss
            running_loss += loss.item()

            if steps % print_every == 0:
                # Model in inference mode, dropout is off
                model.eval()
                
                # Turn off gradients for validation saves memory and computations, so will speed up inference
                with torch.no_grad():
                    valid_loss, valid_accuracy = validation(model, validloader, criterion, device)
                
                # Displaying the validation loss and accuracy during the training. 
                print("Epoch: {}/{}.. ".format(e+1, epochs),
                      "Training Loss: {:.3f}.. ".format(running_loss/print_every),
                      "Validation Loss: {:.3f}.. ".format(valid_loss/len(validloader)),
                      "Validation Accuracy: {:.3f}".format(valid_accuracy/len(validloader)))
                
                running_loss = 0
                
                # Make sure dropout and grads are back on for training
                model.train()

### Load pretrained model and replace classification head

In [ ]:
# Setting all hyper parameters in a dictionary to ease the dealing.
hyper_parameters = {'input_size': 960,       # size of the flattened embedding output of the model
                    'output_size': 102,        # number of classes
                    'hidden_layers': [1024],
                    'drop_p': 0.2,
                    'learn_rate': 1e-4,
                    'epochs': EPOCHS,
                    'model': MODEL_NAME,
                   }

# Creating Feedforward Classifier
classifier = Classifier(
    input_size = hyper_parameters['input_size'], 
    output_size = hyper_parameters['output_size'], 
    hidden_layers = hyper_parameters['hidden_layers'], 
    drop_p = hyper_parameters['drop_p']
)

# Using a Pretrained Network (You may use a different network here, bit we will use mobilenet)
model = models.mobilenet_v3_large(weights='DEFAULT')

# Freezing the parameters so we don't backprop through them, 
# we will backprop through the classifier parameters only later
for param in model.parameters():
    param.requires_grad = FINE_TUNING

# Use the classification network as the classification layer in te pretrained network
model.classifier = classifier

### Loss Function and Optimizer 

For training the model we also need to specify the loss function that should be used as well as the optimizer. In this notebook we will use the Negative Log-Likelihood Loss, which is a special form of the Cross Entropy Loss. For optimizing the model parameters we will use Adam, which works much better for large networks that the standard Stochastic Gradient Descent optimizer. Finally we will print the number of parameters of the model to the console.

In [ ]:
# Define the criterion ( NLLLoss should be used ). 
criterion = nn.NLLLoss()
# Define the optimizer.
optimizer = optim.Adam(model.parameters(), lr=hyper_parameters['learn_rate'])
# Checking if the 'GPU' is available to pass it for the device variable, and if it's not, pass the 'CPU'
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# Count the number of trainable parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Trainable parameters:", total_params)

In [ ]:
# Train the model with a pre-trained network.
training(
    model,
    criterion,
    optimizer,
    device,
    trainloader = dataloaders['trainloader'],
    validloader = dataloaders['validloader'],
    epochs = hyper_parameters['epochs']
)

## Testing your network

It's good practice to test your trained network on test data, images the network has never seen either in training or validation. This will give you a good estimate for the model's performance on completely new images. Run the test images through the network and measure the accuracy, the same way you did validation.

In [ ]:
# Defining a function for testing
def testing(model, testloader):
    ''' Builds a function to test the model, returns the testing accuracy.
        
        Arguments
        ---------
        model: the pre-trained model.
        testloader: generator, the testing dataset.
    '''
    
    # Checking if the 'GPU' is available to pass it for the device variable, and if it's not, pass the 'CPU'
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    # Model in inference mode, dropout is off
    model.eval()
    # Move model to the device
    model.to(device)
    accuracy = 0
    
    for inputs, labels in testloader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        # Turn off gradients for testing saves memory and computations, so will speed up inference
        with torch.no_grad():
            # Forward pass through the network to get the outputs
            output = model.forward(inputs)
        
        # take exponential to get the probabilities from log softmax output.
        ps = torch.exp(output)
        equality = (labels.data == ps.max(dim=1)[1])
        accuracy += equality.type(torch.FloatTensor).mean()
        
    test_accuracy = accuracy / len(testloader)
    
    return test_accuracy

Now we send the testing data through the model. This can take some time when the test dataset is big.

### Task 1
Compute the accuracy for the training and validation datasets and print them to the console.

### Task 2

Compare the accuracy of the test set to the accuracies of the training and validation datasets. Explain if you think your model shows overfitting or not.

In [ ]:
test_accuracy = testing(model, dataloaders['testloader'])

print("Test Accuracy: {:.3f}".format(test_accuracy))

## Save the checkpoint

Now that we have trained the model we want to store the model parameters on the disk, so that we can share it with others and use it in our applications.

In [ ]:
# Saving the model, weights, biases, mapping of classes to indices, and hyper parameters to rebuild the model.

model.class_to_idx = image_datasets['train'].class_to_idx

checkpoint = {'input_size': hyper_parameters['input_size'],
              'output_size': hyper_parameters['output_size'],
              'hidden_layers': hyper_parameters['hidden_layers'],
              'drop_p': hyper_parameters['drop_p'],
              'model': hyper_parameters['model'],
              'class_to_idx': model.class_to_idx,
              'state_dict': model.state_dict()}

filepath = f"{MODEL_NAME}.pth"

torch.save(checkpoint, filepath)

## Loading the checkpoint

In the next cell we show you how you can load model parameters that were saved to disk.

In [ ]:
# Defining a function that loads a checkpoint and rebuilds the model.

def load_checkpoint(filepath):
    ''' Builds a function to load the model, returns the model.
        
        Arguments
        ---------
        filepath: string, the path of the model checkpoint.
    '''
    # Loading the checkpoint
    checkpoint = torch.load(filepath, map_location=torch.device(device))

    # using the pre-trained Network
    if checkpoint['model'] == 'vgg19':
        model = models.vgg19(pretrained=True)
    elif checkpoint['model'] == 'mobilenetv3':
        model = models.mobilenet_v3_large(weights='DEFAULT')
    elif checkpoint['model'] == 'densenet201':
        model = models.densenet201(pretrained=True)
    elif checkpoint['model'] == 'resnet50':
        model = models.resnet50(pretrained=True)
    
    # Freezing the parameters so we don't backprop through them
    for param in model.parameters():
        param.requires_grad = False
       
    # Creating the Feedforward Classifier
    classifier = Classifier(input_size = checkpoint['input_size'],
                         output_size = checkpoint['output_size'],
                         hidden_layers = checkpoint['hidden_layers'], 
                         drop_p = checkpoint['drop_p'])
    
    # Replacing pre-trained calssifier by ours.
    model.classifier = classifier
    # Loading the weights and biases.
    model.load_state_dict(checkpoint['state_dict'])
    # Loading the classes to indices of our saved model.
    model.class_to_idx = checkpoint['class_to_idx']
    
    return model

In [ ]:
# Loading the model & Calculating the spent time.
start = time()
filepath = f"{MODEL_NAME}.pth"
model = load_checkpoint(filepath)
print("Loading model pth time = ", time() - start)

# Inference for classification

Now that we know how to train and save models it is now time to use the model to make predictions (also called inference)

## Image Preprocessing

Before we run images through the model we need to preprocess it, which means reducing the size of the images and normalize the color value. Finally the image need to be converted to a tensor.

In [ ]:
# Defineing a function to process the input image, so it can be used as input for the model

def process_image(image_path):
    ''' Scales, crops, and normalizes a PIL image and converts  into an object 
        that can be used as input to a trained model, returns an Numpy array.

        Arguments
        ---------
        image_path: string, path of the image.
    '''
    
    im = Image.open(image_path)
    # Process a PIL image for use in a PyTorch model
    preprocess = transforms.Compose([transforms.Resize(256),
                                     transforms.RandomResizedCrop(244),
                                     transforms.ToTensor(),
                                     transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                                          std=[0.229, 0.224, 0.225])])
    preprocessed_im = preprocess(im)
    
    return preprocessed_im

To check your work, the function below converts a PyTorch tensor and displays it in the notebook. If your `process_image` function works, running the output through this function should return the original image (except for the cropped out portions).

In [ ]:
def imshow(image, ax=None, title=None):
    """Imshow for Tensor."""
    if ax is None:
        fig, ax = plt.subplots()
    
    # PyTorch tensors assume the color channel is the first dimension
    # but matplotlib assumes is the third dimension
    image = image.numpy().transpose((1, 2, 0))
    
    # Undo preprocessing
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    image = std * image + mean
    
    # Image needs to be clipped between 0 and 1 or it looks like noise when displayed
    image = np.clip(image, 0, 1)
    
    ax.imshow(image)
    
    return ax

In [ ]:
# Load an image and display it
path = './images/flowers/test/1/image_06743.jpg'
Image.open(path)

In [ ]:
# Now load an image and run it through the `process_image` function.
image = process_image(path)
imshow(image);

## Class Prediction

Once you can get images in the correct format, it's time to write a function for making predictions with your model. A common practice is to predict the top 5 or so (usually called top-$K$) most probable classes. You'll want to calculate the class probabilities then find the $K$ largest values.

To get the top $K$ largest values in a tensor use [`x.topk(k)`](http://pytorch.org/docs/master/torch.html#torch.topk). This method returns both the highest `k` probabilities and the indices of those probabilities corresponding to the classes. You need to convert from these indices to the actual class labels using `class_to_idx` which hopefully you added to the model or from an `ImageFolder` you used to load the data ([see here](#Save-the-checkpoint)). Make sure to invert the dictionary so you get a mapping from index to class as well.

Again, this method should take a path to an image and a model checkpoint, then return the probabilities and classes.

```python
probs, classes = predict(image_path, model)
print(probs)
print(classes)
> [ 0.01558163  0.01541934  0.01452626  0.01443549  0.01407339]
> ['70', '3', '45', '62', '55']
```

In [ ]:
# Defining a function to predict the class of the label image.
def predict(image_path, model, topk=5):
    ''' Predict the class (or classes) of an image using a trained deep learning model,
        returns the probabilities and classes of most likely (k) prescited classes.
        
        Arguments
        ---------
        image_path: string, path of the image.
        model: the pre-trained model.
        topk: integer, number of most likely (k) classes.
    '''
    # Move model to the device
    model.to(device)
    # Model in inference mode, dropout is off
    model.eval()
    
    # Load image as tensor with shape torch.Size([3, 244, 244])
    image = process_image(image_path)
    
    # Add a stack dimension, which will result in a tensor shape of torch.Size([1, 3, 244, 244])
    image.unsqueeze_(0) 
    
    # Move image tensors to the device.
    image = image.to(device)
    
    # Turn off gradients for testing saves memory and computations, so will speed up inference.
    with torch.no_grad():
        # Forward pass through the network to get the outputs.
        prediction = model.forward(image)
    # Take exponential to get the probabilities from log softmax output.
    ps = torch.exp(prediction)
    # The most likely (topk) predicted prbabilities with their indices.
    probs, top_k_indices = ps.topk(topk)
    
    # Extracting the classes from the indices.
    classes = []
    for indice in top_k_indices.cpu()[0]:
        classes.append(list(model.class_to_idx)[indice.numpy()]) # Here we take the class from the index.
    
    return probs.cpu()[0].numpy(), classes

### Prediction function

The following function accepts a path to an image, a model, a mapping between numerical category indices to the category names and the number of top classes that should be displayed.

In [ ]:
# Defining a function to map the category labels to category names
def predict_classes_names(image_path, model, cat_to_name, topk=5):
    ''' Mapping the category labels to category names, returns the category names and the probabilities.
        
        Arguments
        ---------
        image_path: string, path of the image.
        model: the pre-trained model.
        cat_to_name: dict, label mapping from category label to category name. 
        topk: integer, number of most likely (k) classes.
    '''
    
    # Obtaining the predicted probabilities and classes from predict function.
    probs, classes = predict(image_path, model, topk)

    # Opening the json file by using with statemnt to be closed after finishing.
    with open(BASE_DIR + '/labels.json', 'r') as f:
        cat_to_name = json.load(f)
    
    # Creating the classes_names list from the classes labels.
    classes_names = []
    for i in classes:
        classes_names.append(cat_to_name[str(i)])
    
    return probs, classes_names

The next function will create a bar chart for the top K classes that the model predicts for a given image.

In [ ]:
def prediction_displaying(image_path, probs, classes_names, cat_to_name):
    ''' Function for displaying the label image and it's most likely predicted class (classes).
    '''
    image = Image.open(image_path)
    
    # Creating axes to subplot
    fig, (ax1, ax2) = plt.subplots(figsize=(6,10), ncols=1, nrows=2)
    
    # Extracting the the class from the path to map it with category name.
    flower_name = cat_to_name[image_path.split('/')[-2]]
    ax1.set_title(flower_name)
    ax1.imshow(image)
    ax1.axis('off')
    
    tick_names = np.arange(len(probs))
    ax2.barh(tick_names, probs, align='center')
    ax2.set_yticks(tick_names)
    ax2.set_yticklabels(classes_names)
    # inverting the element to be displayed from top to bottom.
    ax2.invert_yaxis()
    ax2.set_title('Class Probability')
    
    # Displaying the probability of each class along with bar.
    for i in range(probs.shape[0]):
        count = probs[i]
        pct_string = '{:0.1f}%'.format(count*100)
        plt.text(x = count, y = i, s = pct_string, va = 'center');

Now it's time to load an image and make a prediction.

In [ ]:
# Testing image.
image_path = './images/flowers/test/1/image_06743.jpg'

# Calling predict_classes_names to obtain the most likely 5 predicted classes with their prbabilities.
probs, classes_names = predict_classes_names(image_path, model, cat_to_name, topk=5)

# Calling prediction_displaying to display the image with the predicted classes.
prediction_displaying(image_path, probs, classes_names, cat_to_name)

### Task 3

Take one image from the test dataset and predict the top 5 classes (or less if there are less classes) for that image. Interpret the result.